In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
# stratifiedKFold is a cross-validation technique that ensures that each fold of the dataset has a representative distribution of the target variable. It is particularly useful when dealing with imbalanced datasets, as it helps to maintain the proportion of classes in each fold, leading to more reliable and robust model evaluation.
#GridSearchCV is used to find the best hyperparameters for a model by exhaustively searching through a specified parameter grid. It performs cross-validation to evaluate the performance of each combination of hyperparameters and selects the one that yields the best results based on a specified scoring metric.
from sklearn.pipeline import Pipeline
#Pipeline is a tool in scikit-learn that allows you to chain together multiple steps of a machine learning workflow, such as data preprocessing and model training, into a single object. This helps to streamline the process and ensures that the same transformations are applied consistently during both training and testing phases.
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
#LogisticRegression is a statistical model used for binary classification tasks. It estimates the probability that a given input belongs to a particular class by fitting a logistic function to the data. The model is trained using maximum likelihood estimation, and it can be used to predict the class labels for new, unseen data based on the learned relationships between the features and the target variable.
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
from sklearn.decomposition import PCA
from sklearn.svm import SVC


In [2]:
#configuration

pd.set_option('display.max_columns', None)
#show all columns in the dataframe when printed
pd.set_option('display.float_format', lambda x: '%.3f' % x)
#format floating-point numbers to three decimal places when displayed
pd.set_option('display.max_rows', None)
#show all rows in the dataframe when printed
sns.set_theme(style="darkgrid")
#sets the theme for seaborn plots to "darkgrid", which provides a dark background with gridlines for better visualization of data. 


In [9]:
# loading the data
df=pd.read_csv('../Breast-Cancer-Diagnosis-Analysis/Data Set/Breast Cancer.csv')
df.head(2)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.990,10.380,122.800,1001.000,0.118,0.278,0.300,0.147,0.242,0.079,1.095,0.905,8.589,153.400,0.006,0.049,0.054,0.016,0.030,0.006,25.380,17.330,184.600,2019.000,0.162,0.666,0.712,0.265,0.460,0.119,NaN
1,842517,M,20.570,17.770,132.900,1326.000,0.085,0.079,0.087,0.070,0.181,0.057,0.543,0.734,3.398,74.080,0.005,0.013,0.019,0.013,0.014,0.004,24.990,23.410,158.800,1956.000,0.124,0.187,0.242,0.186,0.275,0.089,NaN


In [10]:
df.shape

(569, 33)

### EDA

In [11]:
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')

In [12]:
df.isnull().sum()

id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_dimension_worst      0
Unnamed:

In [13]:
#dropping the id and unnamed:32 columns as they are not useful for analysis
df.drop(['id','Unnamed: 32'],axis=1,inplace=True)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                569 non-null    object 
 1   radius_mean              569 non-null    float64
 2   texture_mean             569 non-null    float64
 3   perimeter_mean           569 non-null    float64
 4   area_mean                569 non-null    float64
 5   smoothness_mean          569 non-null    float64
 6   compactness_mean         569 non-null    float64
 7   concavity_mean           569 non-null    float64
 8   concave points_mean      569 non-null    float64
 9   symmetry_mean            569 non-null    float64
 10  fractal_dimension_mean   569 non-null    float64
 11  radius_se                569 non-null    float64
 12  texture_se               569 non-null    float64
 13  perimeter_se             569 non-null    float64
 14  area_se                  5

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
#check if there is any encoding missing values - top  repeating values in the dataset
for col in df.columns:
    print(f"Top 5 repeating values in column '{col}':")
    print(df[col].value_counts().head(10))
    print("\n")

Top 5 repeating values in column 'diagnosis':
diagnosis
B    357
M    212
Name: count, dtype: int64


Top 5 repeating values in column 'radius_mean':
radius_mean
12.340    4
11.060    3
10.260    3
12.770    3
13.050    3
13.850    3
12.180    3
11.600    3
13.000    3
11.710    3
Name: count, dtype: int64


Top 5 repeating values in column 'texture_mean':
texture_mean
16.840    3
19.830    3
15.700    3
20.520    3
18.220    3
14.930    3
18.900    3
17.460    3
16.850    3
20.130    2
Name: count, dtype: int64


Top 5 repeating values in column 'perimeter_mean':
perimeter_mean
82.610     3
134.700    3
87.760     3
129.100    2
82.690     2
132.900    2
130.000    2
81.350     2
94.250     2
58.790     2
Name: count, dtype: int64


Top 5 repeating values in column 'area_mean':
area_mean
512.200     3
394.100     2
399.800     2
1076.000    2
582.700     2
334.200     2
1075.000    2
561.000     2
716.600     2
466.100     2
Name: count, dtype: int64


Top 5 repeating values in column

In [17]:
# identyfy the class imbalance
df['diagnosis'].value_counts(normalize=True)*100

diagnosis
B   62.742
M   37.258
Name: proportion, dtype: float64

In [18]:
# map the target variable to binary values
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})